# 03-sql.ipynb

1. 데이터베이스에서 사용 가능한 테이블과 스키마 가져오기
1. 질문과 관련된 테이블을 LLM이 결정
1. 해당 테이블의 스키마 확인하기
1. 질문과 스키마의 정보를 기반으로 쿼리를 생성
1. LLM을 사용하여 흔히 발생하는 오류가 있는지 SQL 확인
1. DB에서 SQL을 실행하고 결과를 확인
1. DB에서 에러 발생시, 수정 후 다시 확인
1. DB 결과를 바탕으로 LLM이 답변 생성

In [2]:
# %pip install langgraph psycopg2-binary

from dotenv import load_dotenv
import os

load_dotenv()



True

In [3]:
from langchain_community.utilities import SQLDatabase
import os

DB_URI = os.environ.get('DB_URI')

db = SQLDatabase.from_uri(DB_URI)

In [4]:
print(db.get_table_info())
print(db.run('SELECT * FROM sales LIMIT 5;'))


CREATE TABLE courses (
	id INTEGER GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1 MINVALUE 1 MAXVALUE 2147483647 CACHE 1 NO CYCLE), 
	name VARCHAR(50), 
	classroom VARCHAR(20), 
	CONSTRAINT courses_pkey PRIMARY KEY (id)
)

/*
3 rows from courses table:
id	name	classroom
1	MySQL 데이터베이스	A관 101호
2	PostgreSQL 고급	B관 203호
3	데이터 분석	A관 704호
*/


CREATE TABLE customers (
	customer_id VARCHAR(10) NOT NULL, 
	customer_name VARCHAR(50) NOT NULL, 
	customer_type VARCHAR(20) NOT NULL, 
	join_date DATE NOT NULL, 
	CONSTRAINT customers_pkey PRIMARY KEY (customer_id)
)

/*
3 rows from customers table:
customer_id	customer_name	customer_type	join_date
C001	김민수	VIP	2023-04-25
C002	이지은	개인	2023-10-09
C003	박서준	개인	2023-08-17
*/


CREATE TABLE dt_demo (
	id INTEGER GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1 MINVALUE 1 MAXVALUE 2147483647 CACHE 1 NO CYCLE), 
	name VARCHAR(20) NOT NULL, 
	nickname VARCHAR(20), 
	birth DATE, 
	score DOUBLE PRECISION, 
	salary NUMERIC(20, 3), 
	descript

In [5]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(name = 'gpt-4.1-mini')

In [6]:
# Agent 용 Tool 만들기

from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x138f1a270>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x138f1a270>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x138f1a270>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [7]:
# Agent 만들기
from langchain.agents import create_agent

dialect = db.dialect
top_k = 5

system_prompt = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
"""

for tool in toolkit.get_tools():
    print(tool.name, tool.description)

sql_db_query Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.
sql_db_schema Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3
sql_db_list_tables Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


In [10]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
# HumanInTheLoop : 언제 멈출 것인지 설정
agent = create_agent(
    model,
    toolkit.get_tools(),
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={'sql_db_query' : True},
            description_prefix= 'Tool 실행 전에 승인을 기다림'
        )
    ],
    checkpointer=InMemorySaver() # 일시정지 - 재실행에서 돌아갈 곳을 기억해야 함!
    
)

In [ ]:
from langgraph.types import Command

question = '2월에 가장 많이 팔린 물건 3개와, 해당 물건들의 토요일 일요일 평균 매출액'

config = {'configurable': {'thread_id': '123456'}}

for event in agent.stream(
    {'messages': [{'role': 'user', 'content': question}]},
    stream_mode='values',
    config=config,
):
    if "__interrupt__" in event: 
        print("INTERRUPTED:") 
        interrupt = event["__interrupt__"][0] 
        for request in interrupt.value["action_requests"]: 
            print(request["description"]) 
    elif "messages" in event:
        event["messages"][-1].pretty_print()
    else:
        pass

print('-----------------------------------------------------------------')

for step in agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}), 
    config,
    stream_mode="values",
):
    if "messages" in step:
        step["messages"][-1].pretty_print()
    elif "__interrupt__" in step:
        print("INTERRUPTED:")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(request["description"])
    else:
        pass

================================ Human Message =================================

전체 평균 매출액과, 가장 구매를 많이한 손님 3명의 이름을 알려줘
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_F28Ryq8lt96tEmIHBwMHFuMp)
 Call ID: call_F28Ryq8lt96tEmIHBwMHFuMp
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

courses, customers, dt_demo, members, practice, sales, students, students_courses, userinfo
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_EzNVOnwqgH4PicQUaDBrNKwa)
 Call ID: call_EzNVOnwqgH4PicQUaDBrNKwa
  Args:
    table_names: sales, customers
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE customers (
	customer_id VARCHAR(10) NOT NULL, 
	customer_name VARCHAR(50) NOT NULL, 
	customer_type VARCHAR(20) NOT NULL, 
	join_date DATE NOT NULL

In [ ]:
1